In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import torchvision.transforms as transforms

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.eval()

# Выберем глубокий слоq layer4[0].conv2.
# Этот слой содержит 512 фильтров. Выберем фильтр с индексом 50
target_layer = model.layer4[0].conv2
target_filter_idx = 50

# Хук для получения активации целевого слоя во время прямого прохода
features = None

def get_features(module, input, output):
    global features
    features = output

# Регистрируем хук
handle = target_layer.register_forward_hook(get_features)

# Инициализация случайного изображения для оптимизации
# ResNet требует размер 224x224
img = torch.randn(1, 3, 224, 224, requires_grad=True)

# Оптимизатор
optimizer = optim.Adam([img], lr=0.1, weight_decay=1e-6)

# Цикл оптимизации
steps = 200
print(f"Максимизация выхода фильтра {target_filter_idx} слоя {target_layer}...")
for i in range(steps):
    optimizer.zero_grad()

    # Прямой проход
    _ = model(img)

    # Получаем активацию
    # features имеет размерность [batch_size, channels, height, width].
    # Берем нужный канал (фильтр) и усредняем по пространственным измерениям.
    target_activation = features[0, target_filter_idx, :, :]

    # Функция потерь: отрицательное среднее значение активации (для максимизации)
    loss = -torch.mean(target_activation)

    loss.backward()
    optimizer.step()

    if (i + 1) % 20 == 0:
        print(f"Step {i+1}, Loss: {loss.item()}")

# Удаляем хук
handle.remove()

# 3. Сохранение изображения
img_result = img.detach().cpu().squeeze(0).permute(1, 2, 0).numpy()
# Нормализация значений пикселей в диапазон [0, 1] для отображения
img_result = (img_result - img_result.min()) / (img_result.max() - img_result.min())

plt.figure(figsize=(6, 6))
plt.imshow(img_result)
plt.axis("off")
plt.title(
    f"Изображение, максимизирующее фильтр {target_filter_idx} слоя {target_layer}"
)
plt.savefig("resnet_maximized_feature.png")
print("Изображение сохранено как resnet_maximized_feature.png")